# Notebook 05 — Results Analysis & Discussion
## Benchmark Comparison, Limitations, and Future Directions

**Author:** Shahid Afridi Laskar  
**Project:** ChemBERTa-ADMET  

---

### Purpose

This notebook consolidates results from all previous notebooks and produces the final comparison figures and discussion. It also connects the ADMET predictions back to the GLUT3 inhibitor project — demonstrating the full CADD pipeline from virtual screening to ADMET filtering.

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.evaluate import compare_models
import warnings
warnings.filterwarnings('ignore')

print('Results analysis notebook ready.')

## 1. Consolidated Results Table

Fill in your results from Notebooks 02 and 03:

In [ ]:
# ── Fill these from your notebook runs ──────────────────────────────────────
results = {
    'RF + ECFP4': {
        'BBBP':  {'ROC_AUC': None, 'PR_AUC': None},
        'ESOL':  {'RMSE': None, 'R2': None},
    },
    'XGBoost + ECFP4': {
        'BBBP':  {'ROC_AUC': None, 'PR_AUC': None},
        'ESOL':  {'RMSE': None, 'R2': None},
    },
    'ChemBERTa (ours)': {
        'BBBP':  {'ROC_AUC': None, 'PR_AUC': None},
        'ESOL':  {'RMSE': None, 'R2': None},
    },
    'MoleculeNet Leaderboard (literature)': {
        'BBBP':  {'ROC_AUC': 0.916, 'PR_AUC': None},
        'ESOL':  {'RMSE': 0.520, 'R2': None},
    },
}

print('Results template ready. Fill in values after running Notebooks 02 and 03.')

## 2. Apply to GLUT3 Inhibitor Candidates

Demonstrating the pipeline on the two lead inhibitors from the published GLUT3 study:

In [ ]:
from src.data_utils import lipinski_filter, smiles_to_rdkit_descriptors

# Placeholder SMILES — replace with actual GLUT3 lead compounds from your paper
glut3_leads = {
    'Lead_1': 'CC1=CC(=CC=C1)NC(=O)C2=CC=C(C=C2)S(=O)(=O)N',
    'Lead_2': 'COC1=CC=C(C=C1)C(=O)NC2=CC=CC=C2F',
}

print('ADMET profile of GLUT3 lead inhibitors:')
print('=' * 60)

rows = []
for name, smi in glut3_leads.items():
    lf = lipinski_filter(smi)
    row = {'Compound': name, 'SMILES': smi}
    row.update(lf)
    rows.append(row)
    print(f'\n{name}:')
    for k, v in lf.items():
        print(f'  {k}: {v}')

df_leads = pd.DataFrame(rows)
print('\nSummary:')
print(df_leads[['Compound', 'MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors', 'TPSA', 'Ro5_pass']].to_string(index=False))

## 3. Radar Chart — Multi-ADMET Profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def radar_chart(categories, values_dict, title='ADMET Radar', save_path=None):
    """Radar/spider chart for multi-property comparison."""
    N = len(categories)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

    for i, (label, values) in enumerate(values_dict.items()):
        values_closed = values + values[:1]
        ax.plot(angles, values_closed, 'o-', linewidth=2, label=label, color=colors[i % len(colors)])
        ax.fill(angles, values_closed, alpha=0.1, color=colors[i % len(colors)])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10)
    ax.set_title(title, size=13, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Normalised ADMET scores (0–1 scale, 1 = best)
# Replace with actual model predictions when trained
categories = ['BBB Permeability', 'Solubility', 'Low Toxicity', 'Lipophilicity OK', 'Ro5 Compliance']

radar_chart(
    categories=categories,
    values_dict={
        'Lead_1': [0.85, 0.60, 0.90, 0.80, 1.0],
        'Lead_2': [0.70, 0.75, 0.85, 0.70, 1.0],
    },
    title='GLUT3 Lead Inhibitors — ADMET Radar',
    save_path='../figures/05_admet_radar.png'
)

## 4. Limitations and Future Work

### Current Limitations

1. **Dataset size**: MoleculeNet datasets are relatively small (~1k–8k compounds). ChemBERTa's advantage over fingerprints grows with dataset size — results here may underestimate its potential on larger proprietary datasets.

2. **Scaffold bias**: Scaffold splitting creates realistic train/test splits but can lead to pessimistic estimates for structurally novel compounds. Activity cliffs remain a challenge.

3. **Calibration**: Raw sigmoid outputs are not calibrated probabilities. Platt scaling or temperature scaling should be applied before using scores for compound prioritisation.

4. **3D information**: ChemBERTa operates on 1D SMILES strings. 3D-aware models (e.g., Uni-Mol, SchNet) may capture pharmacophoric geometry more accurately.

### Future Directions

1. **Extension to GLUT3-specific ADMET filtering**: Apply the trained model to screen the 509 ZINC compounds from the published GLUT3 study — re-ranking candidates by predicted ADMET profile.

2. **TeachOpenCADD talktorial**: Package this pipeline as a talktorial notebook following the TeachOpenCADD format, contributing to the open-source CADD education ecosystem.

3. **HuggingFace deployment**: Deploy the best model to HuggingFace Spaces (Gradio) for zero-install web-based ADMET prediction.

4. **Tox21 multi-task extension**: Extend to all 12 Tox21 endpoints for regulatory-grade toxicity profiling.

5. **Uncertainty quantification**: Add Monte Carlo dropout or deep ensembles to produce prediction intervals — important for prioritising experimental validation.

---

### Connection to Research Goals

This project systematises the ADMET filtering step that was applied manually in the published GLUT3 inhibitor study. By replacing rule-based SwissADME filtering with a learned, multi-task transformer model, we demonstrate that the CADD pipeline can be made more data-driven, reproducible, and extensible — the core philosophy of the TeachOpenCADD framework.